# Objective

The main function is **detect_Bragg_peaks**, whose objective is to look at a given diffraction pattern and extract the (x, y) locations of the peaks. This starts by taking the correlation between the pattern and a 'kernel', which is effectively a snapshot of a vacuum Bragg peak. The correlation output then goes through **get_maxima_2D**, which picks out locations of local maxima (with some minimum specified distance in between them). These local maximum correlations are then fitted with a 2D Gaussian, and the parameters of the fit are used to decide if a maximum is indeed a true Bragg peak or not.

The function **detect_and_save_peaks** will run Bragg peak detection on all diffraction patterns in the dataset and save them as a file.

We will then have two algorithms which preprocess the peaks. One of them (the "regular" algorithm) centers the detected peaks around the vacuum peak, and then extracts those that are near the center (those further away are ignored). The second algorithm (the "dbscanner" algorithm) instead uses the vector differences between each peak and its neighbors, clustering them and using the result instead of the Bragg peaks themselves. (TODO: reword and explain algorithms in more detail later.)

# Imports & data loading

In [ ]:
import py4DSTEM
import numpy as np
import matplotlib.pyplot as plt
from py4DSTEM import show
from scipy.spatial import KDTree
from scipy.ndimage import correlate
from scipy.optimize import curve_fit

py4DSTEM.__version__

In [ ]:
plt.rcParams.update({
    'axes.titlesize': 8,        # Title of the axes
    'axes.labelsize': 8,        # Axis labels
    'xtick.labelsize': 8,       # X-axis tick labels
    'ytick.labelsize': 8,       # Y-axis tick labels
    'legend.fontsize': 8,       # Legend text
    'figure.titlesize': 8,      # Overall figure title (if using suptitle)
    'figure.dpi' : 100
})

In [ ]:
# Load data
data_path = 'file.h5'
datacube  = py4DSTEM.read(data_path)
data      = datacube.data # Numpy array.

In [ ]:
# Check that all intensities are positive.
I_min = np.min(data)
I_min

In [ ]:
## This cell extracts the vacuum peak/probe. We did this in "First look at data.ipynb".

# Select a vacuum region, for making the probe.
mask = np.zeros(datacube.Rshape,dtype=bool)
mask[105:110,60:65] = 1

# Generate a probe.
probe = datacube.get_vacuum_probe( ROI=mask )
alpha_pr,qx0_pr,qy0_pr = py4DSTEM.process.calibration.get_probe_size( probe.probe )

# Prepare the probe kernel.
_ = probe.get_kernel(mode='flat')

In [ ]:
plt.imshow(probe.probe[253:260, 255:262], cmap='grey')
# plt.gca().set_aspect('equal', adjustable='box')
plt.show()

# This looks nice, so let's just choose it as our kernel.
our_kernel = probe.probe[253:260, 255:262] + 0
our_kernel /= np.sum(our_kernel)

# Kernel radius is (roughly) 3 pixels. Min. distance between pixels should be around 6 pixels.

# Bragg peak detection functions

Most of the following functions were copied from [py4DSTEM](https://github.com/py4dstem/py4DSTEM), with modifications to remove fourier/inverse fourier transforms. While doing convolutions/correlations in fourier space is more efficient mathematically, it loses a lot of data because Bragg peaks are sparse (intensities are almost always low values). 

**TODO:** Separately explain all the important hyperparameters. These include arguments to `get_maxima_2D`, initial guesses and lower/upper bounds in `fit_gaussian_2d`, values in `detect_params` dictionary and 'cutoff' in `detection_helper`. 

In [ ]:
def get_maxima_2D(
    ar,
#     subpixel="pixel",
#     upsample_factor=16,
#     sigma=0,
    minAbsoluteIntensity=0,
    minRelativeIntensity=0,
    relativeToPeak=0,
    minSpacing=0,
    edgeBoundary=1,
    maxNumPeaks=1,
#     _ar_FT = None,
):
    """
    Finds the maximal points of a 2D array.

    Args:
        ar (array) the 2D array
        maxNumPeaks: the maximum number of maxima to return
        minAbsoluteIntensity, minRelativeIntensity, relativeToPeak,
            minSpacing, edgeBoundary, maxNumPeaks: filtering applied
            after maximum detection and before subpixel refinement

    Returns:
        a structured array with fields 'x','y','intensity'
    """

    # local pixelwise maxima
    maxima_bool = (
        (ar >= np.roll(ar, (-1, 0), axis=(0, 1)))
        & (ar > np.roll(ar, (1, 0), axis=(0, 1)))
        & (ar >= np.roll(ar, (0, -1), axis=(0, 1)))
        & (ar > np.roll(ar, (0, 1), axis=(0, 1)))
        & (ar >= np.roll(ar, (-1, -1), axis=(0, 1)))
        & (ar > np.roll(ar, (-1, 1), axis=(0, 1)))
        & (ar >= np.roll(ar, (1, -1), axis=(0, 1)))
        & (ar > np.roll(ar, (1, 1), axis=(0, 1)))
    )

    # remove edges
    assert isinstance(edgeBoundary, (int, np.integer))
    if edgeBoundary < 1:
        edgeBoundary = 1
    maxima_bool[:edgeBoundary, :] = False
    maxima_bool[-edgeBoundary:, :] = False
    maxima_bool[:, :edgeBoundary] = False
    maxima_bool[:, -edgeBoundary:] = False

    # Below, np.nonzero returns x/y indices of all local maxima (i.e. 1's in maxima_bool).
    maxima_x, maxima_y = np.nonzero(maxima_bool)
    # Define a structured array and fill it with maxima positions and intensities.
    # NOTE: the intensity values are actual Bragg intensities, NOT correlation values!
    dtype = np.dtype([("x", float), ("y", float), ("intensity", float)])
    maxima = np.zeros(len(maxima_x), dtype=dtype)
    maxima["x"] = maxima_x
    maxima["y"] = maxima_y
    maxima["intensity"] = ar[maxima_x, maxima_y]
    # The following line sorts local maxima by intensity, from HIGHEST to LOWEST.
    maxima = np.sort(maxima, order="intensity")[::-1]

    if len(maxima) == 0:
        return maxima

    # filter
    maxima = filter_2D_maxima(
        maxima,
        minAbsoluteIntensity=minAbsoluteIntensity,
        minRelativeIntensity=minRelativeIntensity,
        relativeToPeak=relativeToPeak,
        minSpacing=minSpacing,
        edgeBoundary=edgeBoundary,
        maxNumPeaks=maxNumPeaks,
    )
    
    return maxima

##############################################################################

def filter_2D_maxima(
    maxima,
    minAbsoluteIntensity=0,
    minRelativeIntensity=0,
    relativeToPeak=0,
    minSpacing=0,
    edgeBoundary=1,
    maxNumPeaks=1,
):
    """
    Args:
        maxima : a numpy structured array with fields 'x', 'y', 'intensity'
        minAbsoluteIntensity : delete counts with intensity below this value
        minSpacing : if two peaks are within this Euclidean distance from one
            another, delete the less intense of the two
        edgeBoundary : delete peaks within this distance of the image edge
        maxNumPeaks : an integer. defaults to 1

    Returns:
        a numpy structured array with fields 'x', 'y', 'intensity'
    """

    # Remove maxima which are too dim
    if minAbsoluteIntensity > 0:
        deletemask = maxima["intensity"] < minAbsoluteIntensity
        maxima = maxima[~deletemask]

    # Below, we remove maxima which are too close. If 2 maxima are within 'minSpacing', we only
    # keep the larger one. Each local maximum is compared to the rest; those which are too close are
    # marked.
    if minSpacing > 0:
        deletemask = np.zeros(len(maxima), dtype=bool)
        for i in range(len(maxima)):
            if deletemask[i] == False:  # noqa: E712
                tooClose = (
                    (maxima["x"] - maxima["x"][i]) ** 2
                    + (maxima["y"] - maxima["y"][i]) ** 2
                ) < minSpacing**2
                tooClose[: i + 1] = False # Peaks of higher/equal intensity will not be deleted.
                deletemask[tooClose] = True
        maxima = maxima[~deletemask]

    # Remove maxima in excess of maxNumPeaks
    if maxNumPeaks is not None:
        if len(maxima) > maxNumPeaks:
            maxima = maxima[:maxNumPeaks]

    return maxima
    
##############################################################################

# Gaussian fitting functions.
def gaussian_2d(coords, amp, x0, y0, sigma_x, offset):
    """
    2D Gaussian function.

    Parameters
    ----------
    coords : tuple of arrays (x, y)
    amp : float
        Amplitude (peak height)
    x0, y0 : float
        Center coordinates
    sigma_x, sigma_y : float
        Standard deviations in x and y
    offset : float
        Constant offset (background)

    Returns
    -------
    flattened 2D Gaussian evaluated on the given coordinates
    """
    x, y = coords
    g = offset + amp * np.exp(
        -(((x - x0) ** 2) / (2 * sigma_x ** 2) + ((y - y0) ** 2) / (2 * sigma_x ** 2))
    )
    return g.ravel()

#  Below: Fits a 2D Gaussian to a 2D numpy array
def fit_gaussian_2d(data):
    """
    Fits a 2D Gaussian to a 2D numpy array.

    Returns
    -------
    popt : array
        Best-fit parameters [amp, x0, y0, sigma_x, sigma_y, offset]
    fit_data : 2D array
        Evaluated Gaussian fit on the same grid as input data
    """

    nrows, ncols = data.shape
    x = np.arange(ncols)
    y = np.arange(nrows)
    x, y = np.meshgrid(x, y)

    # Initial parameter guesses
    amp_guess = np.max(data) - np.min(data)
    x0_guess = 6
    y0_guess = 6
    sigma_x_guess = 2
    offset_guess = np.min(data)

    initial_guess = (amp_guess, x0_guess, y0_guess, sigma_x_guess, offset_guess)

    # Set bounds; Amplitude should be non-negative, deviations should be positive, and
    # offset should also be positive (we made sure of that).
    # NOTE: I'm also bounding peak centers to between (0, 25). I found that scipy attempts to center the peaks
    # outside the box in the fit, which may be why fit algorithm doesn't converge! TODO.
    lower_bounds = (0     , 4.5   , 4.5   , 1  , 0     )
    upper_bounds = (np.inf, 7.5   , 7.5   , 4  , np.inf)
    
    
    # Fit
    popt, pcov = curve_fit(gaussian_2d, (x, y), data.ravel(), p0=initial_guess, bounds=(lower_bounds, upper_bounds))

    # Evaluate fitted Gaussian
    fit_data = gaussian_2d((x, y), *popt).reshape(nrows, ncols)
    return popt, fit_data


# Below: Plots original data and fit
def plot_gaussian_fit(data, fit_data):
    """
    Plots the original 2D data and the fitted Gaussian side by side.
    """

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    im0 = axes[0].imshow(data, origin="upper", aspect='auto')
    axes[0].set_title("Original Data")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(fit_data, origin="upper", aspect='auto')
    axes[1].set_title("Gaussian Fit")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

##############################################################################
grid_size_defined_earlier = 6
detect_params = {'minAbsoluteIntensity': 17, 'minRelativeIntensity': 0,
                 'minSpacing': 10, 'edgeBoundary': grid_size_defined_earlier+1, 'maxNumPeaks': 90}

def detect_Bragg_peaks(diffraction_pattern, probe_template, detect_params=detect_params,
                      cutoff=0.171, return_only_new = True):
    
    # Cross correlation.
    # Check we aren't passing the full 512 x 512 kernel.
    if probe_template.shape == (512, 512):
        print("Error! You are passing the Fourier-space kernel into real-space correlation.")
        print(1/0)

    # Start by making sure the intensities are non-negative.
    min_I = np.min(diffraction_pattern)
    if min_I < 0:
        diffraction_pattern -= min_I

    correlation_result = correlate(diffraction_pattern, probe_template)

    # Find maxima.
    maxima_2D = get_maxima_2D(correlation_result, **detect_params)
    
    # Make it in the form (qx_list, qy_list, I_list), for compatibility. TODO: Use structured array everywhere.
    detected_peaks_OLD = (maxima_2D['x'], maxima_2D['y'], maxima_2D['intensity'])
    
    # Next, we pass the (real-space) correlations and the detected peak indices to a helper function.
    detected_peaks_NEW = detection_helper(correlation_result, detected_peaks_OLD, cutoff=cutoff)
    
    if return_only_new:
        return detected_peaks_NEW
    else:
        return detected_peaks_OLD, detected_peaks_NEW

##############################################################################

grid_size = grid_size_defined_earlier
def detection_helper(correlation, peak_indices_object, cutoff = 0.171): # 0.171 was the smallest ratio for true peaks.
    """
    Checks which of the given peaks pass Gaussian fitting.
    """
    global grid_size, global_popt
    peak_x_list, peak_y_list, peak_I_list = peak_indices_object
    
    # Initialize lists for peaks that pass the Gaussian fit cutoff.
    new_peak_x_list, new_peak_y_list, new_peak_I_list = [], [], []
    
    for peak_x, peak_y, peak_I in zip(peak_x_list, peak_y_list, peak_I_list):
        # Extract a square grid of points centered at (peak_x, peak_y).
        peak_x = int(peak_x)
        peak_y = int(peak_y)
        grid = correlation[peak_x - grid_size : peak_x + grid_size + 1, peak_y - grid_size : peak_y + grid_size + 1]
        
        # Do a Gaussian fit on this grid.
        popt, _ = fit_gaussian_2d(grid)
        
        # Extract 'Amplitude / Offset' value. Reminder: popt = (Amplitude, x0, y0, sigma_x, offset).
        amp_by_offset = popt[0] / popt[-1]
        
        # Save Bragg peak information if it passes the cutoff.
        if amp_by_offset >= cutoff:
            new_peak_x_list.append(peak_x)
            new_peak_y_list.append(peak_y)
            new_peak_I_list.append(peak_I)
            
    # Finally, return the information for these "true" Bragg peaks.
    return (new_peak_x_list, new_peak_y_list, new_peak_I_list)


##############################################################################

def plot_peaks_real(diffraction_pattern, probe_template, cutoff=0.171):
    """Runs Bragg peak detection on the pattern and plots peaks and pattern together.
    Allows user to select a region of diffraction data for Gaussian fitting."""
    global grid_size
    old_peaks, new_peaks = detect_Bragg_peaks(diffraction_pattern, probe_template,
                                              detect_params=detect_params, cutoff=cutoff, return_only_new = False)
    fig, axes = plt.subplots(2, 2, figsize=(5, 5))
    axes = axes.flatten()
    axes[0].imshow(np.log(diffraction_pattern), cmap='grey')
#     axes[0].scatter(old_peaks[1], old_peaks[0], s=200, facecolors='none', edgecolors='red', alpha=0.3, label='old')
    axes[0].scatter(new_peaks[1], new_peaks[0], s=200, facecolors='none', edgecolors='blue', alpha=0.5, label='new')
    axes[0].legend()
    axes[2].set_title("Grid correlation")
    axes[3].set_title("Gaussian fit to grid")
    
    # Below is the callback for the aforementioned GUI.
    selected_pixel = None # Stores relevant Artist object.
    text_object   = None  # Stores the text.
    cbar1 = None
    cbar2 = None
    
    def onclick(event):
        global grid_size
        nonlocal selected_pixel, text_object, cbar1, cbar2
        if event.inaxes != axes[0]:
            return
        # Remember, the 2D 'imshow' plot has rows on y-axis and columns on x-axis. So, we
        # must index as data[y, x].
        mpl_x = int(round(event.xdata))
        mpl_y = int(round(event.ydata))
        np_x  = mpl_y
        np_y  = mpl_x
        
        # We must make sure that matrix indices are within the appropriate range.
        max_x, max_y = diffraction_pattern.shape
        
        if 0 <= np_x <= max_x and 0 <= np_y <= max_y:
            # Extract a grid centered at (np_x, np_y).
            grid = diffraction_pattern[np_x - grid_size: np_x + grid_size + 1,
                                       np_y - grid_size: np_y + grid_size + 1]
            
            # Run correlation on it.
            grid_corr = correlate(grid, probe_template)
            
            # Do a Gaussian fit on this grid's correlation.
            popt, fit_data = fit_gaussian_2d(grid_corr)
            
            # Plot the original grid and the fit_data in appropriate axes.
            # NOTE: axes[1] can be used for fit parameters 🙂.
            text = f"""Peak inds: [{np_x}, {np_y}]
            Amplitude: {popt[0]:.2f}
            Spread: {popt[3]:.2f}
            Offset: {popt[4]:.2f}
            Amplitude / Offset:{popt[0]/popt[4]:.2f}"""
            if text_object:
                text_object.remove()
            text_object = axes[1].text(0.4, 0.5, text, color='black', ha='center', va='center',
                                       transform=axes[1].transAxes, bbox=None)
            _  = axes[2].imshow(grid_corr)
            _1 = axes[3].imshow(fit_data)
            if cbar1:
                cbar1.remove()
            if cbar2:
                cbar2.remove()
            cbar1 = plt.colorbar(_, ax=axes[2])
            cbar2 = plt.colorbar(_1, ax=axes[3])
            
            # Put a green circle around the selected pixel.
            if selected_pixel:
                selected_pixel.remove() # Removes the Artist object.
            selected_pixel = axes[0].scatter([mpl_x], [mpl_y], s=300, facecolors='none', edgecolors='green')
            
        # Update the figure.
        fig.canvas.draw_idle()
    
    fig.canvas.mpl_connect("button_press_event", onclick)
    plt.tight_layout()
    plt.show()
    
    print(f"Number of peaks which pass Gaussian fitting: {len(new_peaks[0])}.")


Run the `plot_peaks_real` GUI to observe Bragg peak detection on an example pattern.

In [ ]:
%matplotlib notebook
sample_pattern = data[83, 127]
plot_peaks_real(sample_pattern, our_kernel, cutoff=0.171)

# Two algorithms

Below we define the two algorithms used for processing the detected peaks, prior to doing crystal phase match analysis. (TODO: This will have to be explained somewhere else, too convoluted for a notebook.) Also, another hyperparameter `global_radius`.

**NOTE:** Both algorithms will center $G\neq 0$ peaks around $G=0$ before returning them. Since Datapoints are calculated from angles about $G=0$ this makes calculations easier, but if we want to plot the results we need to manually re-add the vacuum point's coordinates back to each peak (otherwise they won't overlap with the original diffraction pattern). This is done in `gui`.

Addtionally, note that `array[x, y]` extracts the x-th row and y-th column of an array. However, `ax.imshow([x, y])` plots x on the horizontal axis (column) and y on the vertical (row). So, we need to swap x and y of peaks before plotting them together with the original 2D diffraction pattern (which is plotted using `imshow`).

## 1st algorithm

The first algorithm depends on the correct identification of the G=0 peak within the diffraction pattern. We will use the coordinates of a vacuum peak (peak in a region with no crystal) and declare the nearest peak to be G=0, **with the assertion that it should be no more than 3 pixels away** (an error is thrown otherwise). Furthermore, we would like to save the location of the G=0 peak separately from the rest, for plotting purposes. As such, `get_peaks_near_center` will have 2 outputs (non-center peaks and center peak).

In [ ]:
global_radius = 65
def get_peaks_near_center(peaks, vacuum_peak, rad=global_radius):
    """
    INPUTS
    peaks: Tuple (qx_list, qy_list, I_list) of a single diffraction pattern's peaks.
    vacuum_peak: Tuple (vac_qx, vac_qy) of the vacuum peak k-space coordinates.
    rad: Radius around G=0. Peaks inside it are considered for calculations.
    RETURNS
    chosen_peaks: Tuple (qx_list, qy_list) of peaks centered around G=0.
    vacuum_point: length-2 array; q-coordinates of the G=0 (vacuum) peak.
    """
    
    # Combine the qx,qy-lists into 2D coordinate arrays.
    qx_arr, qy_arr = peaks[0], peaks[1]
    points = np.column_stack( (qx_arr, qy_arr) )
    vac_pt = np.column_stack( (vacuum_peak[0], vacuum_peak[1]) )
    # Make and query the tree to get the vacuum peak in DP.
    tree = KDTree(points)
    dist, vac_ind = tree.query(vac_pt, k=1)

    if dist > 3:
        print(f"Error! Expected small distance to vacuum probe, found {dist}.")
        print(1/1)
        
    # Find the displacement vectors from direct beam to the other Bragg peaks. The
    # subtraction below works correctly, but you can test it to be assured.
    vacuum_point   = points[vac_ind][0] # `[0]` reduces 2D array to 1D, i.e. array([[a, b]]) -> array([a, b]).
    centered_peaks = points - vacuum_point # n x 2 array where each row is a [qx, qy] Bragg peak
    
    chosen_peaks_x, chosen_peaks_y = [], []
    # Iterate over rows (i.e. peaks) of 'centered_peaks'.
    for peak in centered_peaks:
        distance_from_0 = np.linalg.norm(peak)
        if distance_from_0 <= rad and distance_from_0 != 0: # Peak is within the region of interest. Not G=0.
            chosen_peaks_x.append(peak[0])
            chosen_peaks_y.append(peak[1])
            
    chosen_peaks = (chosen_peaks_x, chosen_peaks_y)
            
    return chosen_peaks, vacuum_point

In [ ]:
vacuum_peaks_arr[0,0][0]

## 2nd algorithm

TODO: Explain the additional hyperparameters fed to `dbscanner`. Also, change the function names to something sensible.

In [ ]:
from scipy.spatial import cKDTree

def lattice_vectors(peaks, k=10): # k includes itself and k-1 neighbors.
    """
    Calculates the vector differences between each peak and each of its 
    10 nearest neighbors. We'll use this to extract the lattice basis vectors.
    INPUTS
    peaks : Tuple (qx_arr, qy_arr, I_arr) where entries are arrays/lists of detected peak information
            (q-space position and intensity).
    k     : Number of nearest neighbors used in calculation of lattice vectors.
    OUTPUTS
    vectors: M x 2 array where each entry is a neighbor difference vector.
    """
    # First, we must make sure that there are at least k peaks, or else we'll have an error.
    if len(peaks[0]) < k:
        k = len(peaks[0])
    
    peaks_rearranged = np.column_stack((peaks[0], peaks[1]))
    tree = cKDTree(peaks_rearranged)
    
    # Query all points for their k nearest neighbors. Each point in 'peaks_rearranged' has its own index.
    dists, indices = tree.query(peaks_rearranged, k=k)
#     return peaks_rearranged, dists, indices

    # Compute all neighbor vectors (skip the first column — self)
    vectors = []
    for i in range(len(peaks_rearranged)): # Iterates over each peak.
        for j in range(1, k):  # Iterates over each peak's neighbor's indices. Skip 0th column (the peak itself).
            vec = peaks_rearranged[indices[i, j]] - peaks_rearranged[i]
            vectors.append(vec)

    vectors = np.array(vectors)  # shape (N*(k-1), 2)
    return vectors

#################################################

from sklearn.cluster import DBSCAN
from itertools import combinations

def dbscanner(peaks, eps=3.0, min_samples=8, plot=False, return_metrics=False):
    """
    Given the detected Bragg peaks, finds the vectors to their nearest neighbors, and clusters them
    to extract positions of reciprocal lattice vectors (i.e. cluster centers). Can optionally plot the 
    clusters and return metrics.
    INPUTS
    peaks : N x 2 Numpy array of Bragg peaks detected in a single diffraction pattern.
    OUTPUTS
    cluster_center_tuple : (qx_arr, qy_arr) of lattice vectors found by clustering.
    """
    vectors = lattice_vectors(peaks)
    # 'vectors' will be an empty array if there is only a vacuum peak. In this case, return an empty
    # list (signifying no crystals here).
    if len(vectors) == 0:
        return ([], []) if return_metrics else []
    
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(vectors)

    unique_labels = np.unique(db.labels_)
    # Remove outlier label.
    unique_labels = unique_labels[ unique_labels != -1 ]
    # If we only had outliers, again return nothing.
    if len(unique_labels) == 0:
        return ([], []) if return_metrics else []

    list_of_clusters = [] # Initialize list to store arrays of clusters.

    # Iterate over clusters.
    for label in unique_labels:
        inds = db.labels_ == label
        vectors_in_cluster = vectors[inds]
        list_of_clusters.append(vectors_in_cluster)

    # Find centers and radii of the clusters.
    list_of_cluster_centers = []
    list_of_cluster_radii   = []
    for vectors_in_cluster in list_of_clusters:
        cluster_center = np.mean(vectors_in_cluster, axis=0) # Averages down the columns.
        # Find the maximum distance between a vector in the cluster, and the center.
        max_dist = np.sqrt( np.max( np.sum( (vectors_in_cluster - cluster_center)**2, axis=1 ), axis=0  ) )
        list_of_cluster_centers.append(cluster_center)
        list_of_cluster_radii.append(max_dist)
        
    # If 'return_metrics' is True, we want to calculate the metrics. First, let's find the minimum distance
    # between clusters, as well as the sum of their corresponding radii.
    if return_metrics:
        center_combinations = combinations(list_of_cluster_centers, 2)
        radii_combinations  = combinations(list_of_cluster_radii, 2)
        min_cluster_sep = 99999
        cluster_rad_sum = 0

        for center_pair, radii_pair in zip(center_combinations, radii_combinations):
            current_sep = np.sqrt(np.sum( (center_pair[0] - center_pair[1])**2 ))
            if current_sep < min_cluster_sep:
                min_cluster_sep = current_sep
                cluster_rad_sum = np.sum(radii_pair) # Numpy can sum elements in a tuple.

        # The remaining two metrics are the 'detected_peak_count' and 'cluster_count'.
        detected_peak_count = len(peaks[0])
        cluster_count       = len(list_of_cluster_centers)
        
        # Now, we may finally make a 'lattice_vector_metrics' structured array.
        dtype = np.dtype([("min_cluster_sep"    , float), ("cluster_rad_sum", float),
                          ("detected_peak_count", float), ("cluster_count"  , float)])
        lattice_vector_metrics = np.zeros(1, dtype=dtype)
        lattice_vector_metrics["min_cluster_sep"]     = min_cluster_sep
        lattice_vector_metrics["cluster_rad_sum"]     = cluster_rad_sum
        lattice_vector_metrics["detected_peak_count"] = detected_peak_count
        lattice_vector_metrics["cluster_count"]       = cluster_count
        
#     # Make a plot, if requested.
#     if plot:
#         # Plot each cluster as a different color.
#         fig, ax = plt.subplots()
#         cmap = plt.colormaps['tab20']
#         colors = [cmap(i) for i in range(len(list_of_clusters))]

#         for color, cluster_pts in zip(colors, list_of_clusters):
#             ax.scatter(cluster_pts[:,1], cluster_pts[:,0], color=color)

#         # For reference purpose, add the red circle with radius 60.
#         circle = patches.Circle((0, 0), radius=60, facecolor='none', edgecolor='red')
#         ax.add_patch(circle)
#         ax.set_aspect('equal')
#         ax.grid(visible=True)
#         plt.show()
        
    # Return the list of cluster center positions, as well as metrics if requested.
    # NOTE: Currently, 'list_of_cluster_centers' is a list of 1x2 coordinates. Let's make it instead
    # (qx_list, qy_list) as with previous peaks.
    array_of_cluster_centers = np.array(list_of_cluster_centers)
    les_qxs = array_of_cluster_centers[:,0]
    les_qys = array_of_cluster_centers[:,1]
    cluster_center_tuple = (les_qxs, les_qys)
    
    # Keep only the peaks (cluster centers) near G=0; the rest are redundant and will slow down computation.
    chosen_peaks = get_peaks_near_center_2(cluster_center_tuple)
    
    return (chosen_peaks, lattice_vector_metrics) if return_metrics else chosen_peaks

#################################################

global_radius = 65
def get_peaks_near_center_2(peaks, rad=global_radius):
    """
    INPUTS
    peaks: Tuple (list_of_qx, list_of_qy) of a single diffraction pattern's peaks.
    rad: Radius around G=0. Peaks inside it are considered for calculations
    """

    chosen_peaks_x, chosen_peaks_y = [], []
    # Rearrange the tuple into an N x 2 array of peaks.
    peaks_arr = np.column_stack( (peaks[0], peaks[1]) )
    for peak in peaks_arr:
        distance_from_0 = np.linalg.norm(peak)
        if distance_from_0 <= rad and distance_from_0 != 0: # Peak is within the region of interest. Not G=0.
            chosen_peaks_x.append(peak[0])
            chosen_peaks_y.append(peak[1])
            
    chosen_peaks = (chosen_peaks_x, chosen_peaks_y)

    return chosen_peaks

# Peak detection on the full dataset

Since I'll be revisiting the dataset multiple times, I will run the above peak detection algorithms on every diffraction pattern. This means calling `detect_Bragg_peaks`, `dbscanner` and `get_peaks_near_center` many times.

In [ ]:
def detect_peaks_full(data, probe_template):
    """
    Runs `detect_Bragg_peaks` on the full 4DSTEM dataset.
    INPUTS
    data : 4D Numpy array containing intensity values.
    """
    shape = data.shape    # 122 x 159 x 512 x 512 (as an example).
    shape_2d = shape[0:2] # 122 x 159.
    # Initialize Numpy array where each element will contain info about peaks detected in the corresponding
    # diffraction pattern.
    
    detected_peaks_arr = np.zeros(shape_2d, dtype='object')
    
    # Iterate over all diffraction patterns.
    for i in range(shape_2d[0]):
        for j in range(shape_2d[1]):
            print(f"Current ind: (i, j) = ({i, j})")
            diffraction_pattern = data[i, j]
            
            # Detect peaks in current pattern and put result in array.
            detected_peaks = detect_Bragg_peaks(diffraction_pattern, probe_template) # Is a tuple (qx_list, qy_list).
            detected_peaks_arr[i, j] = detected_peaks
            
    return detected_peaks_arr

In [ ]:
def alg_1_full(input_peaks, input_vac):
    """
    Runs `get_peaks_near_center` (1st algorithm) on the full 4DSTEM dataset.
    INPUTS
    input_peaks : 2D m x n Numpy array calculated by `detect_peaks_full`. (i, j)-th element is a tuple
                  (qx_list, qy_list, I_list) corresponding to q-space positions of peaks in (i, j)-th diffraction pattern.
    input_vac   : (qx, qy)-coordinates of a vacuum peak. This peak is usually found manually.
    OUTPUTS
    array_of_vacuum_peaks : m x n array where the (i, j)-th entry is the vacuum peak's position in the corresponding
                            diffraction pattern.
    array_of_bragg_peaks  : m x n array where the (i, j)-th entry is a tuple (qx_list, qy_list) of selected peaks'
                            q-space positions.
    """
    
    shape = input_peaks.shape
    array_of_bragg_peaks  = np.zeros(shape, dtype='object')
    array_of_vacuum_peaks = np.zeros(shape, dtype='object')

    # Iterate over all real-space positions.
    for i in range(shape[0]):
        for j in range(shape[1]):
            print(f"Current ind: (i, j) = ({i, j})")
            peaks_ij = input_peaks[i, j]
            chosen_peaks, vacuum_point = get_peaks_near_center(peaks_ij, input_vac)
            
            # Put info in appropriate positions in the large arrays.
            array_of_bragg_peaks[i, j]  = chosen_peaks
            array_of_vacuum_peaks[i, j] = vacuum_point
            
    return array_of_bragg_peaks, array_of_vacuum_peaks

In [ ]:
def alg_2_full(array_of_peaks, eps=3.0, min_samples=8):
    """
    Runs `dbscanner` (2nd algorithm) on the full 4DSTEM dataset.
    INPUTS
    array_of_peaks : 2D Numpy array calculated by `detect_peaks_full`. Elements are tuples (qx_list, qy_list, I_list).
    """
    
    shape = array_of_peaks.shape
    dtype = np.dtype([("min_cluster_sep", float), ("cluster_rad_sum", float),
                      ("detected_peak_count", float), ("cluster_count", float)])
    array_of_metrics         = np.zeros(shape, dtype=dtype)
    array_of_lattice_vectors = np.zeros(shape, dtype='object')

    # Now, let's iterate over all the patterns' detected peaks.
    for i in range(shape[0]):
        for j in range(shape[1]):
            print(f"Current ind: (i, j) = ({i, j})")
            peaks = array_of_peaks[i, j]
            list_of_cluster_centers, metrics = dbscanner(peaks, return_metrics=True)
            
            # Put info in appropriate positions in the large arrays. In the case of vacuum peaks, or patterns
            # where there are no clusters, leave array elements as zeros.
            if list_of_cluster_centers:
                array_of_lattice_vectors[i, j] = list_of_cluster_centers
                array_of_metrics[i, j]         = metrics
            
    return array_of_lattice_vectors, array_of_metrics


Let's call these functions on the full dataset and save the results as numpy arrays in the local directory.

In [ ]:
# detected_peaks_arr = detect_peaks_full(data, our_kernel)
# np.save('detected_peaks_arr.npy', detected_peaks_arr, allow_pickle=True)

In [ ]:
detected_peaks_arr[0][0]

To use the 1st algorithm, we need to have the q-coordinates of a vacuum peak. From "First look at data.ipynb", we know that (105, 60) is a region where there's no nanowire. 

In [ ]:
%matplotlib notebook
sample_pattern = data[105, 60]
plot_peaks_real(sample_pattern, our_kernel, cutoff=0.171)

The peak is at (256, 258). Note that there are other pockets of high intensity around this vacuum peak; these are likely due to impurities like carbon.

In [ ]:
# alg_1_peaks_arr, vacuum_peaks_arr = alg_1_full(detected_peaks_arr, (256, 258))
# np.save('alg_1_peaks_arr.npy' , alg_1_peaks_arr , allow_pickle=True)
# np.save('vacuum_peaks_arr.npy', vacuum_peaks_arr, allow_pickle=True)

In [ ]:
alg_1_peaks_arr[0,0]

In [ ]:
vacuum_peaks_arr[0, 0]

Finally, let's do algorithm 2.

In [ ]:
# alg_2_peaks_arr, alg_2_metrics = alg_2_full(detected_peaks_arr)
# np.save('alg_2_peaks_arr.npy', alg_2_peaks_arr, allow_pickle=True)
# np.save('alg_2_metrics.npy'  , alg_2_metrics  , allow_pickle=True)

In [ ]:
alg_2_peaks_arr # It LOOKS like it's just zeroes, hmm...

In [ ]:
alg_2_peaks_arr[alg_2_peaks_arr != 0] # Turns out that isn't always the case!

In [ ]:
alg_2_peaks_arr[alg_2_peaks_arr != 0][4000] # There we have some lattice vectors. 

finished here. next steps

- ~~figure out this broadcast error (maybe dont initialize the 37GiB object array)~~
- gui: include option to show detected peaks. also, modify so that artist.remove is not needed. visibility, updates.
- similarly for algs 1 and 2.
- save alg1, alg2 and detections to disk.

i have 127GB atm, how much after? track; i fear mem leakage

In [ ]:
# Modified gui from "First look at data.ipynb". Now allows us to plot various detected/selected peaks.
def gui(datacube, brightfield):
    data4d      = datacube.data
    m, n = brightfield.shape   
    
    # Plot the brightfield image. It will never be changed/updated.
    fig, (ax_img, ax_dp) = plt.subplots(1, 2, figsize=(10, 4))
    im = ax_img.imshow(brightfield, cmap='grey')
    ax_img.set_title("Brightfield (click a pixel)")
    dp_plot = ax_dp.imshow(np.zeros(data4d.shape[2:]), cmap='gray')
    ax_dp.set_title("Click a pixel → diffraction here")
    
    ## Next, create Artist objects that WILL be changed/updated based on user interactions.
    # 'selected_pixel' will highlight which pixel the user selected on the brightfield image.
    selected_pixel = ax_img.scatter([0], [0], s=1, facecolors='red') # Initialize.
    selected_pixel.set_visible(False)                                # Make invisible until user clicks.
    
    # 'all_detected_peaks' will plot all the peaks detected in the selected diffraction pattern.
    all_detected_peaks = ax_dp.scatter(0, 0, s=200, facecolors='none', edgecolors='blue', alpha=0.7,
                                       label='all detected peaks')
    all_detected_peaks.set_visible(False)
    # 'alg_1_peaks_artist' will plot all peaks selected by the 1st algorithm.
    alg_1_peaks_artist = ax_dp.scatter(0, 0, s=160, facecolors='none', edgecolors='green'  , alpha=0.7, label='alg 1 peaks')
    alg_1_peaks_artist.set_visible(False)
    # 'vacuum_peak_artist' will plot the vacuum peak (also given by the 1st algorithm).
    vacuum_peak_artist = ax_dp.scatter(0, 0, s=160, facecolors='none', edgecolors='purple', alpha=0.7,
                                       label='vacuum peak (G=0)')
    vacuum_peak_artist.set_visible(False)
    # 'alg_2_peaks_artist' will plot all peaks selected by the 2nd algorithm. 
    alg_2_peaks_artist = ax_dp.scatter(0, 0, s=240, facecolors='none', edgecolors='orange', alpha=0.5, label='alg 2 peaks')
    alg_2_peaks_artist.set_visible(False)
    
    # Instantiate plots.
    ax_dp.legend()
    fig.canvas.draw_idle()
    
    def onclick(event):
        nonlocal dp_plot, selected_pixel, all_detected_peaks, alg_1_peaks_artist, alg_2_peaks_artist
        if event.inaxes != ax_img:
            return
        mpl_x = int(round(event.xdata))
        mpl_y = int(round(event.ydata))
        
        # Remember, the 2D 'imshow' plot has rows on y-axis and columns on x-axis. So, we
        # must index as data[y, x].
        np_x = mpl_y
        np_y = mpl_x
        if 0 <= np_x < m and 0 <= np_y < n:
            # Extract and plot the diffraction pattern selected from datacube, in log-scale.
            dp = data4d[np_x, np_y]
            dp_log = np.log(dp)
            dp_plot.set_data(dp_log)
            dp_plot.set_clim(vmin=dp_log.min(), vmax=dp_log.max())
            ax_dp.set_title(f"Diffraction at index [{np_x}, {np_y}]")
            
            # Plot a dot on the selected pixel of the darkfield/boolean map.
            selected_pixel.set_offsets(np.array([[mpl_x, mpl_y]]))
            selected_pixel.set_visible(True)
            
            # Extract all the detected peaks. Plot with x/y reverted.
            peaks_x, peaks_y, _ = detected_peaks_arr[np_x, np_y]
            all_detected_peaks.set_offsets( np.column_stack( (peaks_y, peaks_x) ) )
            all_detected_peaks.set_visible(True) 
            
            # Repeat with the peaks selected by 1st algorithm. Add vacuum offset to each peak.
            peaks_x, peaks_y = alg_1_peaks_arr[np_x, np_y]
            vac_x, vac_y     = vacuum_peaks_arr[np_x, np_y]
            peaks_x += vac_x
            peaks_y += vac_y
            alg_1_peaks_artist.set_offsets( np.column_stack( (peaks_y, peaks_x) ) )
            alg_1_peaks_artist.set_visible(True)
            
            # Plot the vacuum peak.
            vacuum_peak_artist.set_offsets(np.array([[vac_y, vac_x]]))
            vacuum_peak_artist.set_visible(True)         
            
            # Repeat with the peaks selected by 2nd algorithm.
            peaks_x, peaks_y = alg_2_peaks_arr[np_x, np_y]
            peaks_x += vac_x
            peaks_y += vac_y
            alg_2_peaks_artist.set_offsets( np.column_stack( (peaks_y, peaks_x) ) )
            alg_2_peaks_artist.set_visible(True)
            
            # Update the canvas.
            ax_dp.legend()
            fig.canvas.draw_idle()
    
    fig.canvas.mpl_connect('button_press_event', onclick)
    plt.tight_layout()
    plt.show()


In [ ]:
%matplotlib notebook
brightfield = np.load('brightfield.npy', allow_pickle=True)
gui(datacube, brightfield)

**NOTE**: In the original analysis of this dataset, the detect_params had the below form. I used maxnumpeaks=90, which is 40 fewer thn below. This could be a problem.

In [ ]:
detect_params = {'minAbsoluteIntensity': 17, 'minRelativeIntensity': 0,
                 'minSpacing': 10, 'edgeBoundary': grid_size_defined_earlier+1, 'maxNumPeaks': 130}